# 06 | Why do MOD16 and WaPOR disagree in Bagré?
This notebook audits the large difference observed in Notebook 05. It does not train a model. We check original Earth Engine integer values, physical units, time integration, potential ET, and relationships to vegetation and temperature. Nothing overwrites earlier outputs. A modeled ET product is not ground truth.

In [ ]:
%pip -q install earthengine-api rasterio pyproj
import ee, numpy as np, pandas as pd, rasterio
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import date, timedelta
import calendar
from pyproj import Transformer
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Bagre_ET_Water_Stress')
assert ROOT.is_dir(), ROOT
DEST=ROOT/'outputs'/'diagnostics'
DEST.mkdir(parents=True,exist_ok=True)
MONTHS=[f'2021-{m:02d}' for m in range(1,5)]
PROJECT='YOUR_EARTH_ENGINE_PROJECT_ID'
assert PROJECT != 'YOUR_EARTH_ENGINE_PROJECT_ID', 'Set your registered Cloud project ID.'
try:
    ee.Initialize(project=PROJECT)
except ee.EEException:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print('Earth Engine connection:',ee.Number(1).getInfo())
MOD=ee.ImageCollection('MODIS/061/MOD16A2GF')
WAP=ee.ImageCollection('FAO/WAPOR/3/L1_AETI_D')

## 1. Read the common grid and choose diagnostic points
Select three crop-dominated cells with high, medium and low April NDVI. We deliberately sample contrasting locations to inspect behavior, not to estimate population statistics. The MOD16 and WaPOR native pixels covering these coordinates differ in size and location.

In [ ]:
def read(name):
    with rasterio.open(ROOT/'outputs'/f'{name}.tif') as src:
        return src.read(1,masked=True).astype('float64').filled(np.nan),src.transform,src.crs
crop,grid,crs=read('cropland_fraction')
def aligned(name):
    a,t,c=read(name)
    assert a.shape==crop.shape and t==grid and c==crs,name
    return a
ndvi=aligned('ndvi_2021-04')
mod=aligned('mod16_2021-04')
wap=aligned('wapor_2021-04')
valid=(crop>=.5)&np.isfinite(ndvi)&np.isfinite(mod)&np.isfinite(wap)
ids=np.flatnonzero(valid)
assert len(ids)>=3
order=ids[np.argsort(ndvi.ravel()[ids])]
transformer=Transformer.from_crs(crs,'EPSG:4326',always_xy=True)
points=[]
for label,idx in [('low_NDVI',order[0]),('median_NDVI',order[len(order)//2]),('high_NDVI',order[-1])]:
    r,c=np.unravel_index(idx,crop.shape)
    x,y=grid*(c+.5,r+.5)
    lon,lat=transformer.transform(x,y)
    points.append(dict(site=label,row=int(r),col=int(c),lon=lon,lat=lat,
                       april_ndvi=ndvi[r,c],crop_fraction=crop[r,c]))
display(pd.DataFrame(points))
pd.DataFrame(points).to_csv(DEST/'diagnostic_points.csv',index=False)

## 2. Inspect the original source values and metadata
MOD16 ET and PET are eight-day accumulated water depths; multiply raw integers by 0.1 to obtain mm per composite. WaPOR AETI is described by the Earth Engine catalog as mean daily ET including interception for a dekad; multiply raw integers by 0.1 to obtain mm/day. The WaPOR catalog band table labels units as mm, so we explicitly document the description-based interpretation. Both products represent modeled water fluxes, not field measurements.

Sources: [MOD16 user guide](https://lpdaac.usgs.gov/documents/931/MOD16_User_Guide_V61.pdf), [WaPOR v3 catalog](https://developers.google.com/earth-engine/datasets/catalog/FAO_WAPOR_3_L1_AETI_D).

In [ ]:
for label,col,bands in [('MOD16',MOD,['ET','PET','ET_QC']),('WaPOR',WAP,['L1-AETI-D'])]:
    im=ee.Image(col.filterDate('2021-04-01','2021-05-01').first())
    print(label,'bands:',im.bandNames().getInfo())
    print('Native projection:',im.select(bands[0]).projection().getInfo())
    print('Band types:',im.select(bands).bandTypes().getInfo())

## 3. Independent native-pixel time-series audit
Sample each source composite directly at native projection, without the monthly raster export or common-grid resampling. The first MOD16 interval overlapping January can start in December. Keep absent samples missing. We gather raw values, quality flags and original timestamps. The output is small; CPU only.

In [ ]:
fc=ee.FeatureCollection([ee.Feature(ee.Geometry.Point([p['lon'],p['lat']]),{'site':p['site']}) for p in points])
series=[]
for product,col,bands in [('mod16',MOD,['ET','PET','ET_QC']),('wapor',WAP,['L1-AETI-D'])]:
    subset=col.filterDate('2020-12-22','2021-05-01').sort('system:time_start')
    timestamps=subset.aggregate_array('system:time_start').getInfo()
    assert len(timestamps)==len(set(timestamps))
    images=subset.toList(len(timestamps))
    for i,stamp in enumerate(timestamps):
        im=ee.Image(images.get(i)).select(bands)
        projection=im.select(bands[0]).projection()
        stats = im.sampleRegions(
    collection=fc,
    properties=['site'],
    projection=projection,
    geometries=False
).getInfo()
        d=pd.to_datetime(stamp,unit='ms',utc=True).date()
        for f in stats['features']:
            p=f['properties']
            series.append(dict(product=product,date=str(d),site=p['site'],
                               et_raw=p.get(bands[0]),pet_raw=p.get('PET'),et_qc=p.get('ET_QC')))
raw=pd.DataFrame(series)
raw.to_csv(DEST/'native_raw_values.csv',index=False)
display(raw.head(15))

## 4. Recalculate monthly ET from the raw series in pandas
This is a separate implementation from Notebook 03. Multiply by the scale factor exactly once. For MOD16 divide by its actual composite duration to obtain a daily rate; for WaPOR the scaled value is already a daily rate. Multiply by days overlapping each month. Only report monthly totals if all days have valid data. The 25 mm/day guard matches Notebook 03. MOD16 PET is a product-specific modeled potential ET, not FAO-56 ET0.

In [ ]:
def period_end(d,product):
    if product=='mod16': return min(d+timedelta(days=8),date(d.year+1,1,1))
    if d.day==1: return d.replace(day=11)
    if d.day==11: return d.replace(day=21)
    assert d.day==21,d
    return d.replace(day=calendar.monthrange(d.year,d.month)[1])+timedelta(days=1)
def monthly_native(raw):
    result=[]
    for (product,site),g in raw.groupby(['product','site']):
        for m in MONTHS:
            a=date.fromisoformat(m+'-01')
            b=a.replace(day=calendar.monthrange(a.year,a.month)[1])+timedelta(days=1)
            total=potential=0.
            covered=pet_days=0
            for row in g.itertuples():
                d=date.fromisoformat(row.date); e=period_end(d,product)
                w=max(0,(min(e,b)-max(d,a)).days)
                if not w: continue
                factor=.1/(e-d).days if product=='mod16' else .1
                if pd.notna(row.et_raw) and 0<=row.et_raw*factor<=25:
                    total+=row.et_raw*factor*w; covered+=w
                if product=='mod16' and pd.notna(row.pet_raw) and 0<=row.pet_raw*factor<=25:
                    potential+=row.pet_raw*factor*w; pet_days+=w
            expected=(b-a).days
            result.append(dict(product=product,site=site,month=m,days=covered,
                               et_mm=total if covered==expected else np.nan,
                               pet_mm=potential if pet_days==expected else np.nan))
    return pd.DataFrame(result)
assert (period_end(date(2020,12,26),'mod16')-date(2020,12,26)).days==6
assert (period_end(date(2021,2,21),'wapor')-date(2021,2,21)).days==8
native=monthly_native(raw)
native['et_pet']=native.et_mm/native.pet_mm.where(native.pet_mm>0)
for i,row in native.iterrows():
    p=next(p for p in points if p['site']==row.site)
    native.loc[i,'common_grid_et_mm']=aligned(f'{row["product"]}_{row["month"]}')[p['row'],p['col']]
display(native.round(3))
native.to_csv(DEST/'native_monthly_audit.csv',index=False)

A different native value and common-grid value is expected because the latter is an area-averaged raster. Look for the same order of magnitude and seasonal behavior, not exact equality. If the native MOD16 values are also low, the discrepancy is unlikely to be merely a monthly-export artifact.

## 5. Compare both products over identical vegetation-defined subsets
Use cells with at least 50% cropland and paired finite ET estimates. Contrast all such cells with NDVI ≥ 0.25, 0.35 and 0.45. Include coverage and sample sizes so small subsets do not look more reliable than they are. This does not require temperature availability.

In [ ]:
rows=[]
for month in MONTHS:
    x=aligned('mod16_'+month); y=aligned('wapor_'+month)
    n=aligned('ndvi_'+month); mi=aligned('ndmi_'+month); t=aligned('lst_'+month)
    paired=(crop>=.5)&np.isfinite(x)&np.isfinite(y)
    for threshold in [None,.25,.35,.45]:
        keep=paired if threshold is None else paired&(n>=threshold)
        z=x[keep]; q=y[keep]
        rows.append(dict(month=month,ndvi_min='all' if threshold is None else str(threshold),
                         n=len(z),mod16_mean=z.mean() if len(z) else np.nan,
                         wapor_mean=q.mean() if len(q) else np.nan,
                         bias=(q-z).mean() if len(z) else np.nan,
                         rmse=np.sqrt(np.mean((q-z)**2)) if len(z) else np.nan))
subset=pd.DataFrame(rows)
display(subset.round(3))
subset.to_csv(DEST/'active_cropland_comparison.csv',index=False)

## 6. Check spatial associations
Pearson correlations describe relationships, not causal effects. Use the same paired support for each comparison; LST missingness (especially February) changes that support. No significance tests: neighboring cells are spatially dependent. Results with fewer than ten cells or constant values remain missing.

In [ ]:
def corr(x,y,mask):
    keep=mask&np.isfinite(x)&np.isfinite(y)
    a,b=x[keep],y[keep]
    return int(keep.sum()),(np.corrcoef(a,b)[0,1] if len(a)>=10 and a.std()>0 and b.std()>0 else np.nan)
rows=[]
for m in MONTHS:
    x,y=aligned('mod16_'+m),aligned('wapor_'+m)
    paired=(crop>=.5)&np.isfinite(x)&np.isfinite(y)
    for variable in ['ndvi','ndmi','lst']:
        z=aligned(variable+'_'+m)
        for name,a in [('mod16',x),('wapor',y)]:
            n,r=corr(a,z,paired)
            rows.append(dict(month=m,product=name,variable=variable,n=n,r=r))
associations=pd.DataFrame(rows)
display(associations.round(3))
associations.to_csv(DEST/'et_associations.csv',index=False)

## 7. Map numerical disagreement, using identical color limits across months
Positive values mean WaPOR is higher; white means either product is unavailable or the cell has less than 50% cropland. Values are in mm/month. Export the paired difference as GeoTIFF for QGIS.

In [ ]:
from matplotlib.colors import TwoSlopeNorm
diffs={}
for m in MONTHS:
    x,y=aligned('mod16_'+m),aligned('wapor_'+m)
    d=np.where((crop>=.5)&np.isfinite(x)&np.isfinite(y),y-x,np.nan)
    diffs[m]=d
    with rasterio.open(DEST/f'et_difference_{m}.tif','w',driver='GTiff',
                         height=d.shape[0],width=d.shape[1],count=1,dtype='float32',
                         crs=crs,transform=grid,nodata=-9999,compress='deflate') as dst:
        dst.write(np.where(np.isfinite(d),d,-9999).astype('float32'),1)
finite=np.concatenate([d[np.isfinite(d)] for d in diffs.values()])
assert finite.size
lim=max(1,float(np.max(np.abs(finite))))
fig,axes=plt.subplots(1,4,figsize=(14,4),constrained_layout=True)
for ax,(m,d) in zip(axes,diffs.items()):
    p=ax.imshow(d,cmap='RdBu',norm=TwoSlopeNorm(vmin=-lim,vcenter=0,vmax=lim))
    ax.set_title(m); ax.axis('off')
fig.colorbar(p,ax=axes,label='WaPOR minus MOD16 (mm/month)',shrink=.7)
fig.savefig(ROOT/'figures'/'et_product_difference.png',dpi=180)
plt.show()

## Checkpoint
Share the native monthly audit table, active cropland comparison, correlations and the difference map. We can then judge whether the product-level discrepancy persists after independent sampling. Do not infer an irrigation amount or a validated water stress diagnosis. No deep learning is needed to answer this research question.